# Gen Python API — Walkthrough

This notebook demonstrates the core capabilities of the `gen` Python package:
initializing a repository, importing a reference sequence, applying variants
from a VCF, searching for sequence motifs, and visualizing the resulting graph.

In [1]:
import os
import tempfile

import gen

print(f"gen version: {gen.__version__}")

gen version: 0.1.31


## Initialize a repository

A `Repository` is the top-level object. Passing a path to a `.gen` directory
creates a new repository there; omitting the path searches the current working
directory for an existing one.

In [2]:
tmpdir = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmpdir, "gen"))
print(f"Repository created at: {repo.gen_dir}")

Repository created at: /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/tmpabrgfnjs/gen/.gen


## Import a reference sequence

`import_reference_fasta` loads a FASTA file and registers each contig under a
named reference. The reference name (e.g. `"hg38"`) is how you refer to it
when applying variants or querying the repository.

In [3]:
fasta_path = os.path.join(tmpdir, "reference.fa")
with open(fasta_path, "w") as f:
    f.write(">m123\nATCGATCGATCGATCGATCGGGAACACACAGAGA\n")

result = repo.import_reference_fasta(fasta_path, "reference")
print(result)

'/var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/tmpabrgfnjs/reference.fa' imported.


## Explore block groups

Each contig in the imported FASTA becomes a *block group* — the internal
representation of a sequence graph.

In [4]:
bgs = repo.get_block_groups()
print(f"{len(bgs)} block group(s) after import:")
for bg in bgs:
    print(f"  name={bg.name!r}  collection={bg.collection_name!r}  sample={bg.sample_name!r}")

1 block group(s) after import:
  name='m123'  collection='default'  sample='reference'


## Apply variants from a VCF

`update_with_vcf` reads a VCF file and creates a new sample whose graph
incorporates the specified variants. The `sample` argument selects which
sample column to read from the VCF.

In [5]:
vcf_path = os.path.join(tmpdir, "variants.vcf")
with open(vcf_path, "w") as f:
    f.write(
        "##fileformat=VCFv4.1\n"
        "##contig=<ID=m123,length=34>\n"
        "##FORMAT=<ID=GT,Number=1,Type=String,Description=\"Genotype\">\n"
        "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tsample1\n"
        "m123\t3\t.\tCG\tC\t.\t.\t.\tGT\t1/1\n"
        "m123\t10\t.\tT\tTAGA\t.\t.\t.\tGT\t1/1\n"
    )

result = repo.update_with_vcf(vcf_path, sample="sample1", reference="reference")
print(result)

Updated from '/var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/tmpabrgfnjs/variants.vcf'.


In [6]:
bgs = repo.get_block_groups()
print(f"{len(bgs)} block group(s) after VCF update:")
for bg in bgs:
    print(f"  name={bg.name!r}  collection={bg.collection_name!r}  sample={bg.sample_name!r}")

2 block group(s) after VCF update:
  name='m123'  collection='default'  sample='reference'
  name='m123'  collection='default'  sample='sample1'


## Search for a sequence motif

`repo.search(query)` finds all occurrences of a sequence across every block
group. Results are `(BlockGroup, [GraphLocus])` pairs; each `GraphLocus`
describes where in the graph the match lives.

In [7]:
query = "ATCGATCG"
results = repo.search(query)

print(f"Searching for {query!r}:")
for bg, loci in results:
    if loci:
        print(f"  {bg.name!r} (sample={bg.sample_name!r}): {len(loci)} match(es)")
        for locus in loci:
            print(f"    strand={locus.strand}  blocks={len(locus.slices)}")

Searching for 'ATCGATCG':
  'm123' (sample='reference'): 7 match(es)
    strand=+  blocks=1
    strand=+  blocks=1
    strand=+  blocks=1
    strand=+  blocks=1
    strand=-  blocks=1
    strand=-  blocks=1
    strand=-  blocks=1
  'm123' (sample='sample1'): 7 match(es)
    strand=+  blocks=3
    strand=+  blocks=2
    strand=+  blocks=2
    strand=+  blocks=1
    strand=-  blocks=3
    strand=-  blocks=2
    strand=-  blocks=1


## Visualize the graph

`bg.plot()` returns an interactive `GenGraphWidget`. This requires the Jupyter
extra (`pip install gen[jupyter]`). In a notebook environment it renders an
interactive terminal-style graph viewer you can navigate with the mouse.

In [8]:
reference_bg = next(bg for bg in bgs if bg.sample_name == "reference")

if gen.GenGraphWidget is not None:
    widget = reference_bg.plot(cols=120)
    display(widget)
else:
    print("Jupyter widget not available. Install with: pip install gen[jupyter]")